In [ ]:
from collections import Counter
from pathlib import Path
import sys

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from data.datasets import get_dataset
from data.splits import get_train_val_split

dataset_name = "Oxford-IIIT Pet classification"
data_dir = repo_root / "data" / "raw" / "iit_pet"

# Oxford-IIIT Pet provides official trainval and test partitions.
trainval_dataset = get_dataset(
    split="trainval",
    dataset=dataset_name,
    data_dir=data_dir,
    download=True,
)
test_dataset = get_dataset(
    split="test",
    dataset=dataset_name,
    data_dir=data_dir,
)

train_dataset, val_dataset = get_train_val_split(
    trainval_dataset,
    ratio=0.2,
    seed=42,
    labels=trainval_dataset._labels,
)

for name, subset in (("Train", train_dataset), ("Validation", val_dataset)):
    counts = Counter(trainval_dataset._labels[i] for i in subset.indices)
    print(
        f"{name}: {len(subset)} images, {len(counts)} classes "
        f"({min(counts.values())}-{max(counts.values())} per class)"
    )

print(f"Test: {len(test_dataset)} images (official held-out set)")
image, label = train_dataset[0]
print(f"Example: image size={image.size}, label={label}")


 16%|█▌        | 125M/792M [06:36<1:06:44, 167kB/s]  

In [ ]:
from functools import partial
from torch.utils.data import Subset

from data.datasets import calculate_mean_std
from data.dataloaders import get_dataloader
from data.transforms import image_transform

# Calculate normalization statistics from raw TRAINING images only.
mean, std = calculate_mean_std(train_dataset, size=(224, 224))
print(f"Training RGB mean: {mean}")
print(f"Training RGB std:  {std}")

train_transform = partial(
    image_transform, mean=mean, std=std, train=True, size=(224, 224)
)
eval_transform = partial(
    image_transform, mean=mean, std=std, train=False, size=(224, 224)
)

# Both instances use the same original file ordering. Keep the indices
# already produced by the stratified split in the first cell.
train_indices = train_dataset.indices
val_indices = val_dataset.indices

train_dataset = Subset(
    get_dataset(
        split="trainval",
        dataset=dataset_name,
        data_dir=data_dir,
        transform=train_transform,
    ),
    train_indices,
)
val_dataset = Subset(
    get_dataset(
        split="trainval",
        dataset=dataset_name,
        data_dir=data_dir,
        transform=eval_transform,
    ),
    val_indices,
)
test_dataset = get_dataset(
    split="test",
    dataset=dataset_name,
    data_dir=data_dir,
    transform=eval_transform,
)

train_loader = get_dataloader(train_dataset, split="train", batch_size=16, num_workers=0)
val_loader = get_dataloader(val_dataset, split="val", batch_size=16, num_workers=0)
test_loader = get_dataloader(test_dataset, split="test", batch_size=16, num_workers=0)

for name, loader in (("train", train_loader), ("val", val_loader), ("test", test_loader)):
    images, labels = next(iter(loader))
    print(f"{name}: images={tuple(images.shape)}, labels={tuple(labels.shape)}")
